<a href="https://colab.research.google.com/github/SalmaAnindhia/PemMes_25_Salma/blob/main/JS09/JS09_Tugas2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q annoy
!pip install -q faiss-cpu
!pip install -q hnswlib
!pip install -q kagglehub

In [4]:
import os, glob
import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

In [5]:
import kagglehub
path = kagglehub.dataset_download("bwandowando/spotify-songs-with-attributes-and-lyrics")

csv_files = sorted(glob.glob(os.path.join(path, "**", "*.csv"), recursive=True))
for i, f in enumerate(csv_files):
    print(i, os.path.basename(f), round(os.path.getsize(f) / 1e6, 1), "MB")

100%|██████████| 894M/894M [00:09<00:00, 95.2MB/s]

Extracting files...


0 songs_with_attributes_and_lyrics.csv 1545.1 MB
1 songs_with_lyrics_and_timestamps.csv 2420.5 MB


In [6]:
PILIH = 0
df = pd.read_csv(csv_files[PILIH])
print(csv_files[PILIH], df.shape)
print(df.columns.tolist())
df.head()

/root/.cache/kagglehub/datasets/bwandowando/spotify-songs-with-attributes-and-lyrics/versions/20/songs_with_attributes_and_lyrics.csv (955320, 17)
['id', 'name', 'album_name', 'artists', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'lyrics']


,id,name,album_name,artists,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,lyrics
0,0Prct5TDjAnEgIqbxcldY9,!,UNDEN!ABLE,HELLYEAH,0.415,0.6050,7,-11.157,1,0.0575,0.00116,0.838000,0.4710,0.193,100.059,79500.0,"He said he came from Jamaica,\n he owned a cou..."
1,2ASl4wirkeYm3OWZxXKYuq,!!,NaN,Yxngxr1,0.788,0.6480,7,-9.135,0,0.3150,0.90000,0.000000,0.1760,0.287,79.998,114000.0,"Fucked a bitch, now she running with my kids\n..."
2,69lcggVPmOr9cvPx9kLiiN,!!! - Interlude,Where I Belong EP,Glowie,0.000,0.0354,7,-20.151,0,0.0000,0.90800,0.000000,0.4790,0.000,0.000,11413.0,"Oh, my God, I'm going crazy\n"
3,4U7dlZjg1s9pjdppqZy0fm,!!De Repente!!,Un Palo Al Agua (20 Grandes Canciones),Rosendo,0.657,0.8820,5,-6.340,1,0.0385,0.00740,0.000013,0.0474,0.939,123.588,198173.0,Continuamente se extraña la gente si no puede ...
4,4v1IBp3Y3rpkWmWzIlkYju,!!De Repente!!,Fuera De Lugar,Rosendo,0.659,0.8930,5,-8.531,1,0.0411,0.09220,0.000019,0.0534,0.951,123.600,199827.0,Continuamente se extraña la gente si no puede ...


In [8]:
candidate = ['danceability', 'energy', 'loudness', 'speechiness',
             'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
features = [c for c in candidate if c in df.columns]
print("Fitur dipakai:", features)
print("Tidak ditemukan:", set(candidate) - set(features))

df = df.dropna(subset=features).reset_index(drop=True)
X = df[features].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype('float32')

name_col = next((c for c in ['track_name', 'name', 'song', 'title'] if c in df.columns), None)
artist_col = next((c for c in ['artist_name', 'artists', 'artist'] if c in df.columns), None)
print("Kolom nama:", name_col, "| kolom artis:", artist_col)
print("Shape X_scaled:", X_scaled.shape)

k = 10

Fitur dipakai: ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
Tidak ditemukan: set()
Kolom nama: name | kolom artis: artists
Shape X_scaled: (955320, 9)


In [10]:
def show_tracks(idx_list):
    cols = [c for c in [name_col, artist_col] if c]
    return df.loc[list(idx_list), cols] if cols else pd.DataFrame({"index": list(idx_list)})

q_idx = 10
query_vec = X_scaled[q_idx:q_idx+1]
print("Track query:")
display(show_tracks([q_idx]))

def drop_self(idx_row, q):
    return [int(i) for i in idx_row if int(i) != q][:k]

Track query:


,name,artists
10,!liaF cipE,One Morning Left


In [11]:
nn = NearestNeighbors(n_neighbors=k+1, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)

start = time.time()
dist_ex, idx_ex = nn.kneighbors(query_vec)
time_exact = (time.time() - start) * 1000
res_exact = drop_self(idx_ex[0], q_idx)

print("Exact NN index:", res_exact)
print("Waktu:", round(time_exact, 3), "ms")
show_tracks(res_exact)

Exact NN index: [95943, 949528, 949529, 388811, 512885, 632115, 615740, 198356, 821580, 95591]
Waktu: 248.508 ms


,name,artists
95943,Blood out of Stone,Prong
949528,白銀,LiSA
949529,白銀,LiSA
388811,In the Darkest Night,"[""Autumn's Grey Solace""]"
512885,Missing,Flyleaf
632115,Rain Falls Down,We The Kings
615740,Pray for Rain,Josephine Collective
198356,Do Nothing,Prong
821580,Top of the World,Greek Fire
95591,Blood Of My Enemy,Winds of Plague


In [12]:
dim = X_scaled.shape[1]

start = time.time()
annoy_index = AnnoyIndex(dim, 'euclidean')
for i, v in enumerate(X_scaled):
    annoy_index.add_item(i, v)
annoy_index.build(10)
build_annoy = time.time() - start

start = time.time()
idx_an = annoy_index.get_nns_by_vector(query_vec[0], k+1)
time_annoy = (time.time() - start) * 1000
res_annoy = drop_self(idx_an, q_idx)

print("ANNOY index:", res_annoy)
print("Build:", round(build_annoy, 2), "detik | Query:", round(time_annoy, 3), "ms")
show_tracks(res_annoy)

ANNOY index: [95943, 949528, 949529, 388811, 512885, 632115, 615740, 198356, 821580, 95591]
Build: 25.24 detik | Query: 0.372 ms


,name,artists
95943,Blood out of Stone,Prong
949528,白銀,LiSA
949529,白銀,LiSA
388811,In the Darkest Night,"[""Autumn's Grey Solace""]"
512885,Missing,Flyleaf
632115,Rain Falls Down,We The Kings
615740,Pray for Rain,Josephine Collective
198356,Do Nothing,Prong
821580,Top of the World,Greek Fire
95591,Blood Of My Enemy,Winds of Plague


In [13]:
n = X_scaled.shape[0]
nlist = max(1, min(int(np.sqrt(n)), n // 39))

start = time.time()
quantizer = faiss.IndexFlatL2(dim)
faiss_index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_L2)
faiss_index.train(X_scaled[:min(n, 200_000)])
faiss_index.add(X_scaled)
build_faiss = time.time() - start

faiss_index.nprobe = min(10, nlist)
start = time.time()
_, idx_fa = faiss_index.search(query_vec, k+1)
time_faiss = (time.time() - start) * 1000
res_faiss = drop_self(idx_fa[0], q_idx)

print("FAISS index:", res_faiss)
print("Build:", round(build_faiss, 2), "detik | Query:", round(time_faiss, 3), "ms")
show_tracks(res_faiss)

FAISS index: [95943, 949528, 949529, 388811, 512885, 632115, 615740, 198356, 821580, 95591]
Build: 3.04 detik | Query: 5.109 ms


,name,artists
95943,Blood out of Stone,Prong
949528,白銀,LiSA
949529,白銀,LiSA
388811,In the Darkest Night,"[""Autumn's Grey Solace""]"
512885,Missing,Flyleaf
632115,Rain Falls Down,We The Kings
615740,Pray for Rain,Josephine Collective
198356,Do Nothing,Prong
821580,Top of the World,Greek Fire
95591,Blood Of My Enemy,Winds of Plague


In [14]:
start = time.time()
hnsw_index = hnswlib.Index(space='l2', dim=dim)
hnsw_index.init_index(max_elements=n, ef_construction=100, M=16)
hnsw_index.add_items(X_scaled)
build_hnsw = time.time() - start

hnsw_index.set_ef(50)
start = time.time()
idx_hn, _ = hnsw_index.knn_query(query_vec, k+1)
time_hnsw = (time.time() - start) * 1000
res_hnsw = drop_self(idx_hn[0], q_idx)

print("HNSW index:", res_hnsw)
print("Build:", round(build_hnsw, 2), "detik | Query:", round(time_hnsw, 3), "ms")
show_tracks(res_hnsw)

HNSW index: [95943, 949528, 949529, 388811, 512885, 632115, 615740, 198356, 821580, 95591]
Build: 107.7 detik | Query: 0.302 ms


,name,artists
95943,Blood out of Stone,Prong
949528,白銀,LiSA
949529,白銀,LiSA
388811,In the Darkest Night,"[""Autumn's Grey Solace""]"
512885,Missing,Flyleaf
632115,Rain Falls Down,We The Kings
615740,Pray for Rain,Josephine Collective
198356,Do Nothing,Prong
821580,Top of the World,Greek Fire
95591,Blood Of My Enemy,Winds of Plague


In [15]:
def overlap(res, gt):
    return len(set(res) & set(gt)) / len(gt)

compare = pd.DataFrame({
    "Metode": ["Exact NN", "ANNOY", "FAISS (IVF)", "HNSW"],
    "Build (detik)": [0, build_annoy, build_faiss, build_hnsw],
    "Query (ms)": [time_exact, time_annoy, time_faiss, time_hnsw],
    "Recall@10 (vs Exact)": [1.0, overlap(res_annoy, res_exact),
                             overlap(res_faiss, res_exact), overlap(res_hnsw, res_exact)],
})
display(compare.round(4))

,Metode,Build (detik),Query (ms),Recall@10 (vs Exact)
0,Exact NN,0.0000,248.5085,1.0
1,ANNOY,25.2401,0.3719,1.0
2,FAISS (IVF),3.0354,5.1088,1.0
3,HNSW,107.7046,0.3018,1.0


In [16]:
n_eval = 200
rng = np.random.default_rng(42)
q_ids = rng.choice(n, n_eval, replace=False)
Qs = X_scaled[q_ids]

t0 = time.time()
_, I_ex = nn.kneighbors(Qs)
t_ex = (time.time() - t0) * 1000 / n_eval
gt = [drop_self(r, q) for r, q in zip(I_ex, q_ids)]

# ANNOY
t0 = time.time()
I_an = [annoy_index.get_nns_by_vector(v, k+1) for v in Qs]
t_an = (time.time() - t0) * 1000 / n_eval
pr_an = [drop_self(r, q) for r, q in zip(I_an, q_ids)]

# FAISS
t0 = time.time()
_, I_fa = faiss_index.search(Qs, k+1)
t_fa = (time.time() - t0) * 1000 / n_eval
pr_fa = [drop_self(r, q) for r, q in zip(I_fa, q_ids)]

# HNSW
t0 = time.time()
I_hn, _ = hnsw_index.knn_query(Qs, k+1)
t_hn = (time.time() - t0) * 1000 / n_eval
pr_hn = [drop_self(r, q) for r, q in zip(I_hn, q_ids)]

def rec(pred, gt):
    return np.mean([len(set(p) & set(g)) / len(g) for p, g in zip(pred, gt)])

final = pd.DataFrame({
    "Metode": ["Exact NN", "ANNOY", "FAISS (IVF)", "HNSW"],
    "Build (detik)": [0, build_annoy, build_faiss, build_hnsw],
    "Query rata-rata (ms)": [t_ex, t_an, t_fa, t_hn],
    "Recall@10": [1.0, rec(pr_an, gt), rec(pr_fa, gt), rec(pr_hn, gt)],
})
final.round(4)

,Metode,Build (detik),Query rata-rata (ms),Recall@10
0,Exact NN,0.0000,4.5950,1.0000
1,ANNOY,25.2401,0.1059,0.8275
2,FAISS (IVF),3.0354,0.1416,0.9930
3,HNSW,107.7046,0.1381,0.9980
